In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

In [ ]:
np.random.seed(42)
X, _ = make_blobs(n_samples=300, centers=[[30,20],[60,60],[85,85]],
                  cluster_std=[6,7,6], random_state=42)
noise = np.array([[10,90],[95,10],[50,5],[5,50],[100,50]])
X = np.vstack([X, noise])
df = pd.DataFrame(X, columns=['Annual_Income_k$','Spending_Score'])
print('Dataset shape:', df.shape)
df.head()

In [ ]:
print(df.describe())
plt.figure(figsize=(6,4))
plt.scatter(df['Annual_Income_k$'], df['Spending_Score'],
            alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score')
plt.title('Raw Customer Data')
plt.tight_layout()
plt.savefig('raw_data.png', dpi=120)
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print('Mean after scaling:', X_scaled.mean(axis=0).round(4))
print('Std  after scaling:', X_scaled.std(axis=0).round(4))

In [ ]:
inertias, sil_scores, K_range = [], [], range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(K_range, inertias, 'bo-')
ax1.axvline(3, color='red', linestyle='--', label='K=3')
ax1.set_title('Elbow Method'); ax1.set_xlabel('K'); ax1.set_ylabel('Inertia'); ax1.legend()
ax2.plot(K_range, sil_scores, 'go-')
ax2.axvline(3, color='red', linestyle='--', label='K=3')
ax2.set_title('Silhouette Scores'); ax2.set_xlabel('K'); ax2.set_ylabel('Score'); ax2.legend()
plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=120)
plt.show()
print('Best K:', K_range[sil_scores.index(max(sil_scores))])

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
df['KMeans_Cluster'] = kmeans_labels
colors = ['#E74C3C','#2ECC71','#3498DB']

plt.figure(figsize=(8,6))
for i, color in enumerate(colors):
    mask = kmeans_labels == i
    plt.scatter(X_scaled[mask,0], X_scaled[mask,1], c=color,
                label=f'Cluster {i}', alpha=0.7, edgecolors='white', s=60)
centroids = kmeans.cluster_centers_
plt.scatter(centroids[:,0], centroids[:,1], c='black', marker='X', s=200, zorder=5, label='Centroids')
plt.xlabel('Income (scaled)'); plt.ylabel('Spending (scaled)')
plt.title('K-Means Clustering (K=3)'); plt.legend()
plt.tight_layout()
plt.savefig('kmeans_clusters.png', dpi=120)
plt.show()
print('Silhouette Score:', round(silhouette_score(X_scaled, kmeans_labels), 4))
print('Cluster sizes:', dict(zip(*np.unique(kmeans_labels, return_counts=True))))

In [ ]:
nbrs = NearestNeighbors(n_neighbors=5).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
distances = np.sort(distances[:, 4])

plt.figure(figsize=(7,4))
plt.plot(distances, color='purple')
plt.axhline(y=0.5, color='red', linestyle='--', label='eps=0.5')
plt.xlabel('Points (sorted)'); plt.ylabel('5th NN Distance')
plt.title('k-Distance Graph'); plt.legend()
plt.tight_layout()
plt.savefig('kdistance.png', dpi=120)
plt.show()

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)
df['DBSCAN_Cluster'] = dbscan_labels

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
print(f'Clusters: {n_clusters} | Noise points: {n_noise}')

unique_labels = sorted(set(dbscan_labels))
palette = ['#E74C3C','#2ECC71','#3498DB','#F39C12','#9B59B6']
color_map = {-1: '#AAAAAA'}
for i, lbl in enumerate([l for l in unique_labels if l != -1]):
    color_map[lbl] = palette[i % len(palette)]

plt.figure(figsize=(8,6))
for label in unique_labels:
    mask = dbscan_labels == label
    lname = f'Cluster {label}' if label != -1 else 'Noise'
    marker = 'x' if label == -1 else 'o'
    plt.scatter(X_scaled[mask,0], X_scaled[mask,1], c=color_map[label],
                label=lname, alpha=0.8, marker=marker,
                edgecolors='white' if label != -1 else 'none', s=60)
plt.xlabel('Income (scaled)'); plt.ylabel('Spending (scaled)')
plt.title('DBSCAN (eps=0.5, min_samples=5)'); plt.legend()
plt.tight_layout()
plt.savefig('dbscan_clusters.png', dpi=120)
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,6))
for i, color in enumerate(colors):
    mask = kmeans_labels == i
    ax1.scatter(X_scaled[mask,0], X_scaled[mask,1], c=color,
                label=f'Cluster {i}', alpha=0.7, edgecolors='white', s=55)
ax1.scatter(centroids[:,0], centroids[:,1], c='black', marker='X', s=200, zorder=5, label='Centroids')
ax1.set_title('K-Means (K=3)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Income (scaled)'); ax1.set_ylabel('Spending (scaled)'); ax1.legend()

for label in sorted(set(dbscan_labels)):
    mask = dbscan_labels == label
    lname = f'Cluster {label}' if label != -1 else 'Noise'
    marker = 'x' if label == -1 else 'o'
    ax2.scatter(X_scaled[mask,0], X_scaled[mask,1], c=color_map[label],
                label=lname, alpha=0.8, marker=marker,
                edgecolors='white' if label != -1 else 'none', s=55)
ax2.set_title('DBSCAN (eps=0.5, min_samples=5)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Income (scaled)'); ax2.set_ylabel('Spending (scaled)'); ax2.legend()

plt.suptitle('K-Means vs DBSCAN', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('comparison_plot.png', dpi=120)
plt.show()

In [ ]:
km_sil = silhouette_score(X_scaled, kmeans_labels)
valid_mask = dbscan_labels != -1
db_sil = silhouette_score(X_scaled[valid_mask], dbscan_labels[valid_mask]) if len(set(dbscan_labels[valid_mask])) > 1 else None

print('='*50)
print('         EVALUATION SUMMARY')
print('='*50)
print(f'K-Means Silhouette Score : {km_sil:.4f}')
print(f'DBSCAN  Silhouette Score : {db_sil:.4f}' if db_sil else 'DBSCAN Silhouette: N/A')
print(f'K-Means Inertia          : {kmeans.inertia_:.2f}')
print(f'DBSCAN Clusters Found    : {n_clusters}')
print(f'DBSCAN Noise Points      : {n_noise}')
print('='*50)

summary = pd.DataFrame({
    'Metric':  ['Algorithm','Clusters','Noise Points','Silhouette','Inertia'],
    'K-Means': ['K-Means','3','0 (all assigned)',f'{km_sil:.4f}',f'{kmeans.inertia_:.2f}'],
    'DBSCAN':  ['DBSCAN',str(n_clusters),str(n_noise),f'{db_sil:.4f}' if db_sil else 'N/A','N/A']
})
print(summary.to_string(index=False))